In [1]:
!pip install -q evaluate bitsandbytes trl rouge_score bert_score sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.2 MB/s eta 0:00:00


In [2]:
# !pip install -q flash-attn --no-build-isolation

In [3]:
!pip install -q qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 54.1 MB/s eta 0:00:00


In [4]:
import torch
import json
import gc
import numpy as np
from tqdm import tqdm
from collections import Counter
from datasets import load_dataset
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from qwen_vl_utils import process_vision_info
import evaluate
from bert_score import score as bert_score_fn
import string

# --- 1. CẤU HÌNH ---
base_model_id = "Qwen/Qwen3-VL-2B-Instruct"
test_dataset_path = "/kaggle/input/datasets/thanhngphu/test-vizwiz-transformed/test_dataset.json"

# Đổi tên file output để tách biệt với kết quả fine-tune
output_file = "/kaggle/working/inference_evaluation_results_base.json"

SYSTEM_PROMPT = """
Bạn là một trợ lý AI xuất sắc trong việc phân tích hình ảnh (VQA).
Nhiệm vụ của bạn là trả lời câu hỏi của người dùng dựa trên hình ảnh được cung cấp một cách ngắn gọn, chính xác bằng Tiếng Việt một cách tự nhiên.

QUY TẮC QUAN TRỌNG:
1. Bạn phải CỐ GẮNG HẾT SỨC để trả lời dựa trên bất kỳ chi tiết nào bạn có thể nhận diện được trong ảnh,một cách ngắn gọn , xúc tích.
2. CHỈ KHI hình ảnh THỰC SỰ QUÁ MỜ, nhòe, nhiễu nặng hoặc tối đen đến mức MỘT CON NGƯỜI cũng không thể đoán được bất cứ thông tin gì liên quan đến câu hỏi,thì bạn phải trả lời đúng một câu: "Ảnh mờ, bạn vui lòng chụp lại ảnh rõ hơn."
3. Nếu ảnh chỉ hơi mờ hoặc thiếu sáng nhưng vẫn có thể suy luận được, tuyệt đối không phàn nàn, hãy đưa ra câu trả lời tốt nhất của bạn.
4. Trả lời ngắn gọn, súc tích trong một câu, không giải thích dài dòng, không thêm thông tin không cần thiết, chỉ trả lời đúng trọng tâm câu hỏi.
5. Luôn trả lời bằng tiếng Việt, không được trả lời bằng ngôn ngữ khác, không được trả lời bằng tiếng Anh, không được trả lời bằng tiếng Trung, chỉ được trả lời bằng tiếng Việt.
"""

In [5]:
# --- 2. LOAD BASE MODEL VÀ PROCESSOR ---
print(f"Đang load Processor từ {base_model_id}...")
processor = AutoProcessor.from_pretrained(base_model_id, trust_remote_code=True)

print(f"Đang load Base Model ({base_model_id}) bfloat16...")
model = Qwen3VLForConditionalGeneration.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    # attn_implementation="flash_attention_2" # Bật lên nếu môi trường hỗ trợ
)
model.eval()

Đang load Processor từ Qwen/Qwen3-VL-2B-Instruct...


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Đang load Base Model (Qwen/Qwen3-VL-2B-Instruct) bfloat16...


model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

Qwen3VLForConditionalGeneration(
  (model): Qwen3VLModel(
    (visual): Qwen3VLVisionModel(
      (patch_embed): Qwen3VLVisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3VLVisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3VLVisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3VLVisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
          )
        )
      )
 

In [6]:
# --- 3. INFERENCE TỪNG MẪU (BATCH SIZE = 1) ---
print("Đang tải tập dữ liệu test...")
test_data = load_dataset("json", data_files=test_dataset_path, split="train")

predictions = []
references = []
full_results = []

print(f"Bắt đầu Inference từng ảnh trên {len(test_data)} mẫu...")

for item in tqdm(test_data):
    image_path = item['images'][0]
    question = item['messages'][0]['content'].replace("<image>", "").strip()
    ground_truth = item['messages'][1]['content']
    
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {
                "type": "image", 
                "image": image_path,
                "max_pixels": 1003520 # Giữ ở mức HD (1 triệu pixel) để đủ nét mà không nổ VRAM
            },
            {"type": "text", "text": question}
        ]}
    ]
    
    # Tiền xử lý
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    # Generate
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=128)
    
    # Giải mã
    generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

    predictions.append(output_text)
    references.append(ground_truth)
    
    full_results.append({
        "image": image_path.split("/")[-1],
        "question": question,
        "ground_truth": ground_truth,
        "prediction": output_text
    })
    
    # Xóa rác giải phóng VRAM ngay sau mỗi ảnh
    del inputs, generated_ids, image_inputs, video_inputs
    torch.cuda.empty_cache()
    gc.collect()


Đang tải tập dữ liệu test...


Generating train split: 0 examples [00:00, ? examples/s]

Bắt đầu Inference từng ảnh trên 200 mẫu...


100%|██████████| 200/200 [1:42:23<00:00, 30.72s/it]


In [7]:
# --- 4. TÍNH TOÁN ĐỘ ĐO ---
def compute_f1_overlap(prediction, ground_truth):
    p_tokens, g_tokens = prediction.lower().split(), ground_truth.lower().split()
    common = Counter(p_tokens) & Counter(g_tokens)
    num_same = sum(common.values())
    if num_same == 0: return 0
    p, r = 1.0 * num_same / len(p_tokens), 1.0 * num_same / len(g_tokens)
    return (2 * p * r) / (p + r)

def compute_exact_match(prediction, ground_truth):
    def normalize(text):
        return text.lower().strip().translate(str.maketrans('', '', string.punctuation))
    return int(normalize(prediction) == normalize(ground_truth))

print("\nĐang tính toán các độ đo NLP...")
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")

bleu_results = bleu.compute(predictions=predictions, references=[[r] for r in references])
rouge_results = rouge.compute(predictions=predictions, references=references)
meteor_results = meteor.compute(predictions=predictions, references=references)

f1_scores = [compute_f1_overlap(p, r) for p, r in zip(predictions, references)]
avg_f1_overlap = np.mean(f1_scores)

em_scores = [compute_exact_match(p, r) for p, r in zip(predictions, references)]
avg_em_score = np.mean(em_scores)

print("Đang tính BERTScore...")
P, R, F1_bert = bert_score_fn(predictions, references, lang="vi", verbose=False)
avg_bert_score = F1_bert.mean().item()


Đang tính toán các độ đo NLP...


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


Đang tính BERTScore...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
# --- 5. LƯU KẾT QUẢ ---
print("\n" + "="*40)
print(f"KẾT QUẢ ĐÁNH GIÁ (BASE MODEL - {len(test_data)} MẪU)")
print("="*40)
print(f"Exact Match: {avg_em_score:.4f}")
print(f"F1-Overlap:  {avg_f1_overlap:.4f}")
print(f"BERTScore:   {avg_bert_score:.4f}")
print(f"BLEU Score:  {bleu_results['bleu']:.4f}")
print(f"ROUGE-L:     {rouge_results['rougeL']:.4f}")
print(f"METEOR:      {meteor_results['meteor']:.4f}")
print("="*40)

final_output = {
    "metrics": {
        "exact_match": avg_em_score, "f1_overlap": avg_f1_overlap, "bert_score": avg_bert_score,
        "bleu": bleu_results['bleu'], "rougeL": rouge_results['rougeL'], "meteor": meteor_results['meteor']
    },
    "details": full_results
}

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(final_output, f, ensure_ascii=False, indent=4)
print(f"Đã lưu kết quả BASE MODEL tại: {output_file}")


KẾT QUẢ ĐÁNH GIÁ (BASE MODEL - 200 MẪU)
Exact Match: 0.0800
F1-Overlap:  0.3045
BERTScore:   0.7662
BLEU Score:  0.1158
ROUGE-L:     0.3964
METEOR:      0.2840
Đã lưu kết quả BASE MODEL tại: /kaggle/working/inference_evaluation_results_base.json
